# MVP: Previsão de Total de Medalhas — Olimpíadas de Verão

Este notebook implementa um pipeline completo de Machine Learning para prever o total de medalhas por delegação nas Olimpíadas de Verão, utilizando features como PIB (GDP), nível de renda, ano e histórico de participação.

**Objetivo:** Construir e comparar modelos de regressão (Ridge, Random Forest, XGBoost) e encontrar o melhor desempenho.

## 1. Setup: Importações e Configurações

In [ ]:
# Fixar seed para reprodutibilidade
RANDOM_STATE = 42

# Bibliotecas principais
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
from scipy.stats import randint, uniform
import warnings
warnings.filterwarnings('ignore')

# Tentar importar xgboost; instalar se necessário
try:
    import xgboost as xgb
except Exception:
    !pip install xgboost -q
    import xgboost as xgb

print("✓ Setup concluído!")

## 2. Carregar e Explorar Dados

In [ ]:
# CORRIGIDO: URL deve apontar para raw.githubusercontent.com
DATA_URL = 'https://raw.githubusercontent.com/thiagolmetne/Mvp---Machine-Learning-Analytics/main/gpd_medalha-000000000000.csv'

if DATA_URL:
    try:
        df = pd.read_csv(DATA_URL)
        print(f"✓ Dados carregados com sucesso! Shape: {df.shape}")
    except Exception as e:
        print(f"✗ Erro ao carregar de {DATA_URL}: {e}")
        print("\nTentando upload manual no Colab...")
        try:
            from google.colab import files
            uploaded = files.upload()
            fname = list(uploaded.keys())[0]
            df = pd.read_csv(io.BytesIO(uploaded[fname]))
            print(f"✓ CSV enviado manualmente! Shape: {df.shape}")
        except:
            print("Nenhum arquivo disponível.")
            df = None
else:
    print("DATA_URL não definida.")
    df = None

if df is not None:
    print("\nPrimeiras linhas:")
    display(df.head())

## 3. Análise Exploratória (EDA)

In [ ]:
if df is not None:
    # Info e estatísticas
    print("=" * 60)
    print("INFO DO DATAFRAME")
    print("=" * 60)
    display(df.info())
    
    print("\n" + "=" * 60)
    print("ESTATÍSTICAS DESCRITIVAS")
    print("=" * 60)
    display(df.describe(include='all').T)
    
    print("\n" + "=" * 60)
    print("VALORES FALTANTES")
    print("=" * 60)
    print(df.isna().sum())
    
    # Distribuição do target
    print("\n" + "=" * 60)
    print("DISTRIBUIÇÃO DO TARGET (total_medalhas)")
    print("=" * 60)
    plt.figure(figsize=(10, 5))
    sns.histplot(df['total_medalhas'].fillna(0), bins=50, kde=True)
    plt.title('Distribuição de total_medalhas')
    plt.xlabel('total_medalhas')
    plt.ylabel('Frequência')
    plt.grid(alpha=0.3)
    plt.show()
    
    # Valores categóricos únicos
    print("\n" + "=" * 60)
    print("VALORES ÚNICOS (Categóricos)")
    print("=" * 60)
    print(f"nivel_renda únicas: {df['nivel_renda'].unique()}")
    print(f"\nDelegações (top 15):")
    print(df['delegacao'].value_counts().head(15))

## 4. Limpeza e Preparação de Dados

In [ ]:
if df is not None:
    # Fazer cópia para não modificar original
    df_clean = df.copy()
    
    # Converter colunas para tipos apropriados
    df_clean['ano'] = df_clean['ano'].astype(int)
    for col in ['media_gdp', 'total_medalhas', 'qtd_paises']:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    # Remover linhas sem delegacao ou sem ano
    df_clean = df_clean.dropna(subset=['delegacao', 'ano']).reset_index(drop=True)
    
    print(f"Shape após limpeza: {df_clean.shape}")
    print(f"Missing values por coluna:\n{df_clean.isna().sum()}")
    print(f"\nAnos disponíveis: {sorted(df_clean['ano'].unique())}")

## 5. Feature Engineering

**Estratégia:**
- Imputar `nivel_renda` faltante como 'Unknown'
- Transformar `media_gdp` com `log1p()` para lidar com assimetria
- Codificar delegação: manter top N (por soma de medalhas) como one-hot, agrupar resto como 'OUTROS'
- Usar `ano` e `qtd_paises` como features numéricas

In [ ]:
if df is not None:
    TOP_N_DELEGACOES = 15  # Manter top 15 delegações
    
    # Step 1: Imputar nivel_renda faltante
    df_clean['nivel_renda'] = df_clean['nivel_renda'].fillna('Unknown')
    
    # Step 2: Criar feature de log(gdp)
    if 'media_gdp' in df_clean.columns:
        df_clean['log_media_gdp'] = np.log1p(df_clean['media_gdp'].fillna(0))
    
    # Step 3: Agrupar delegações menos frequentes
    top_delegacoes = df_clean['delegacao'].value_counts().head(TOP_N_DELEGACOES).index.tolist()
    df_clean['delegacao_grouped'] = df_clean['delegacao'].apply(
        lambda x: x if x in top_delegacoes else 'OUTROS'
    )
    
    print(f"✓ Feature Engineering concluído!")
    print(f"\nDelegações mantidas (top {TOP_N_DELEGACOES}):")
    print(df_clean['delegacao_grouped'].value_counts())

## 6. Time-Split: Treino e Teste

**Estratégia:** Usar time-split para simular previsão realista:
- **Treino:** Todos os anos anteriores ao último
- **Teste:** Registros do último ano

In [ ]:
if df is not None:
    # Time-split
    max_year = df_clean['ano'].max()
    min_year = df_clean['ano'].min()
    
    train_data = df_clean[df_clean['ano'] < max_year].copy()
    test_data = df_clean[df_clean['ano'] == max_year].copy()
    
    print(f"Years range: {min_year} - {max_year}")
    print(f"\nTrain set: anos < {max_year}")
    print(f"  Shape: {train_data.shape}")
    print(f"  Years: {sorted(train_data['ano'].unique())}")
    print(f"\nTest set: ano = {max_year}")
    print(f"  Shape: {test_data.shape}")
    
    # Definir features e target
    feature_cols = ['ano', 'delegacao_grouped', 'nivel_renda', 'log_media_gdp', 'qtd_paises']
    target_col = 'total_medalhas'
    
    # Remover linhas com target faltante
    train_data = train_data.dropna(subset=[target_col])
    test_data = test_data.dropna(subset=[target_col])
    
    X_train = train_data[feature_cols].copy()
    y_train = train_data[target_col].copy()
    X_test = test_data[feature_cols].copy()
    y_test = test_data[target_col].copy()
    
    # Guardar dados originais do test set para análise posterior
    test_df = test_data[['delegacao', 'ano', 'nivel_renda', 'media_gdp', 'total_medalhas']].copy()
    
    print(f"\n✓ Time-split concluído!")
    print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")

## 7. Construir Pipelines de Pré-processamento e Modelos

In [ ]:
if df is not None:
    # Definir features numéricas e categóricas
    num_features = ['ano', 'log_media_gdp', 'qtd_paises']
    cat_features = ['delegacao_grouped', 'nivel_renda']
    
    # Pipeline de pré-processamento
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), num_features),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
                ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), cat_features)
        ]
    )
    
    # Dicionário com modelos
    models = {
        'dummy': Pipeline([
            ('preprocessor', preprocessor),
            ('model', DummyRegressor(strategy='mean'))
        ]),
        'ridge': Pipeline([
            ('preprocessor', preprocessor),
            ('model', Ridge(alpha=1.0, random_state=RANDOM_STATE))
        ]),
        'rf': Pipeline([
            ('preprocessor', preprocessor),
            ('model', RandomForestRegressor(n_estimators=100, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1))
        ]),
        'xgb': Pipeline([
            ('preprocessor', preprocessor),
            ('model', xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=RANDOM_STATE))
        ])
    }
    
    print("✓ Pipelines criados!")
    print(f"Modelos: {list(models.keys())}")

## 8. Função Auxiliar: Avaliar Modelos

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, cv=4):
    """
    Avaliar modelo em treino e teste.
    Retorna dicionário com métricas e predições.
    """
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calcular métricas
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')
    cv_mae_mean = -cv_scores.mean()
    cv_mae_std = cv_scores.std()
    
    return {
        'train_mae': train_mae,
        'test_mae': test_mae,
        'test_rmse': test_rmse,
        'test_r2': test_r2,
        'cv_mae_mean': cv_mae_mean,
        'cv_mae_std': cv_mae_std,
        'preds': y_test_pred
    }

print("✓ Função evaluate_model definida!")

## 9. Treinar e Avaliar Modelos Base

In [ ]:
if df is not None:
    results = {}
    
    for model_name, model_pipe in models.items():
        print(f"\n{'='*60}")
        print(f"Treinando: {model_name.upper()}")
        print(f"{'='*60}")
        
        # Treinar
        model_pipe.fit(X_train, y_train)
        
        # Avaliar
        result = evaluate_model(model_pipe, X_train, y_train, X_test, y_test, cv=4)
        results[model_name] = result
        
        # Exibir resultados
        print(f"Train MAE:     {result['train_mae']:.4f}")
        print(f"Test MAE:      {result['test_mae']:.4f}")
        print(f"Test RMSE:     {result['test_rmse']:.4f}")
        print(f"Test R²:       {result['test_r2']:.4f}")
        print(f"CV MAE:        {result['cv_mae_mean']:.4f} (±{result['cv_mae_std']:.4f})")
    
    # Resumo comparativo
    print(f"\n{'='*60}")
    print("RESUMO COMPARATIVO")
    print(f"{'='*60}")
    
    res_df = pd.DataFrame([
        {
            'model': name,
            'train_mae': res['train_mae'],
            'test_mae': res['test_mae'],
            'test_rmse': res['test_rmse'],
            'test_r2': res['test_r2'],
            'cv_mae': f"{res['cv_mae_mean']:.4f} (±{res['cv_mae_std']:.4f})"
        }
        for name, res in results.items()
    ]).sort_values('test_mae')
    
    display(res_df)
    
    best_model_name = res_df.iloc[0]['model']
    print(f"\n✓ Melhor modelo (por test MAE): {best_model_name.upper()}")

## 10. Tuning de Hiperparâmetros: XGBoost com RandomizedSearchCV

**Nota:** Esta célula pode demorar alguns minutos.

In [ ]:
if df is not None:
    print("Iniciando RandomizedSearchCV para XGBoost...")
    print("(Esta célula pode demorar alguns minutos)\n")
    
    xgb_pipe = models['xgb']
    
    # Distribuições de hiperparâmetros
    param_dist = {
        'model__n_estimators': randint(50, 300),
        'model__max_depth': randint(3, 10),
        'model__learning_rate': uniform(0.01, 0.3),
        'model__subsample': uniform(0.6, 0.4),
        'model__colsample_bytree': uniform(0.5, 0.5),
        'model__min_child_weight': randint(1, 5)
    }
    
    # RandomizedSearchCV
    search = RandomizedSearchCV(
        xgb_pipe,
        param_distributions=param_dist,
        n_iter=20,  # Reduzido para 20 para executar mais rápido
        scoring='neg_mean_absolute_error',
        cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1
    )
    
    search.fit(X_train, y_train)
    
    best_xgb = search.best_estimator_
    print(f"\n✓ Best params encontrados:")
    for key, value in search.best_params_.items():
        print(f"  {key}: {value}")
    print(f"\n  Best CV MAE: {-search.best_score_:.4f}")
    
    # Avaliar melhor modelo
    best_res = evaluate_model(best_xgb, X_train, y_train, X_test, y_test, cv=3)
    results['xgb_tuned'] = best_res
    
    print(f"\nXGBoost Tuned Test MAE: {best_res['test_mae']:.4f}")
    print(f"XGBoost Tuned Test RMSE: {best_res['test_rmse']:.4f}")
    print(f"XGBoost Tuned Test R²: {best_res['test_r2']:.4f}")

## 11. Comparação Final e Seleção do Melhor Modelo

In [ ]:
if df is not None:
    print(f"{'='*60}")
    print("COMPARAÇÃO FINAL - TODOS OS MODELOS")
    print(f"{'='*60}\n")
    
    final_res_df = pd.DataFrame([
        {
            'model': name,
            'train_mae': res['train_mae'],
            'test_mae': res['test_mae'],
            'test_rmse': res['test_rmse'],
            'test_r2': res['test_r2']
        }
        for name, res in results.items()
    ]).sort_values('test_mae').reset_index(drop=True)
    
    display(final_res_df)
    
    best_model_name = final_res_df.iloc[0]['model']
    print(f"\n{'='*60}")
    print(f"✓ MELHOR MODELO: {best_model_name.upper()}")
    print(f"{'='*60}")
    print(f"Test MAE:  {final_res_df.iloc[0]['test_mae']:.4f}")
    print(f"Test RMSE: {final_res_df.iloc[0]['test_rmse']:.4f}")
    print(f"Test R²:   {final_res_df.iloc[0]['test_r2']:.4f}")

## 12. Visualizações: Predito vs Real

In [ ]:
if df is not None:
    preds = results[best_model_name]['preds']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Scatter
    axes[0].scatter(y_test, preds, alpha=0.6, s=100, edgecolors='k', linewidth=0.5)
    axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfeito')
    axes[0].set_xlabel('Medalhas Reais', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Medalhas Preditas', fontsize=11, fontweight='bold')
    axes[0].set_title(f'Predito vs Real — {best_model_name.upper()}', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Plot 2: Resíduos
    residuals = y_test.values - preds
    axes[1].scatter(preds, residuals, alpha=0.6, s=100, edgecolors='k', linewidth=0.5)
    axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
    axes[1].set_xlabel('Predições', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Resíduos', fontsize=11, fontweight='bold')
    axes[1].set_title('Análise de Resíduos', fontsize=12, fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Tabela com predições
    print("\nPrincipais Predições (ordenadas por predição):")
    out_df = test_df.copy()
    out_df['pred'] = preds
    out_df['erro'] = np.abs(out_df['total_medalhas'] - out_df['pred'])
    display(out_df.sort_values('pred', ascending=False).head(15))

## 13. Feature Importance (XGBoost)

In [ ]:
if df is not None and 'xgb' in best_model_name:
    print("Extraindo Feature Importance...\n")
    
    # Preparar nomes de features
    preprocessor.fit(X_train)
    
    # Feature names numéricos
    num_names = num_features
    
    # Feature names categóricos (após one-hot encoding)
    ohe = preprocessor.named_transformers_['cat'].named_steps['ohe']
    cat_names = ohe.get_feature_names_out(cat_features).tolist()
    
    feature_names = num_names + cat_names
    
    # Extrair modelo do pipeline
    if best_model_name == 'xgb_tuned':
        model_obj = best_xgb.named_steps['model']
    else:
        model_obj = models['xgb'].named_steps['model']
    
    # Feature importance
    try:
        importances = model_obj.feature_importances_
        fi_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)
        
        # Plot
        plt.figure(figsize=(10, 6))
        fi_series.head(20).plot(kind='barh', color='steelblue')
        plt.xlabel('Importância', fontsize=11, fontweight='bold')
        plt.title('Feature Importance (Top 20) — XGBoost', fontsize=12, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print("\nTop 10 Features:")
        display(fi_series.head(10))
    except Exception as e:
        print(f"Não foi possível extrair feature importance: {e}")
else:
    if df is not None:
        print(f"Feature importance disponível apenas para modelos XGBoost.")
        print(f"Melhor modelo: {best_model_name}")

## 14. Salvar Modelo

In [ ]:
if df is not None:
    # Selecionar melhor modelo completo
    if best_model_name == 'xgb_tuned':
        final_model = best_xgb
    elif best_model_name in models:
        final_model = models[best_model_name]
    else:
        final_model = None
    
    if final_model is not None:
        # Treinar no dataset completo (treino + teste) para máxima informação
        X_combined = pd.concat([X_train, X_test], ignore_index=True)
        y_combined = pd.concat([y_train, y_test], ignore_index=True)
        final_model.fit(X_combined, y_combined)
        
        # Salvar
        model_filename = 'mvp_olimpiadas_best_model.joblib'
        joblib.dump(final_model, model_filename)
        print(f"✓ Modelo salvo: {model_filename}")
        print(f"  Melhor modelo: {best_model_name}")
        print(f"  Test MAE: {final_res_df.iloc[0]['test_mae']:.4f}")
    else:
        print("✗ Nenhum modelo disponível para salvar.")

## 15. Conclusão e Próximos Passos

### Resumo do Projeto

✓ **O que foi feito:**
- Análise exploratória de dados (EDA) completa
- Limpeza, pré-processamento e feature engineering
- Validação temporal (time-split): treino em anos passados, teste no ano mais recente
- Construção de 4 pipelines (Dummy, Ridge, Random Forest, XGBoost)
- Tuning de hiperparâmetros com RandomizedSearchCV
- Avaliação com métricas: MAE, RMSE, R²
- Visualizações e análise de resíduos
- Feature importance (XGBoost)
- Salvamento do melhor modelo

### Resultados Principais
```
Melhor Modelo: [PREENCHIDO AUTOMATICAMENTE]
Test MAE:     [PREENCHIDO AUTOMATICAMENTE]
Test RMSE:    [PREENCHIDO AUTOMATICAMENTE]
Test R²:      [PREENCHIDO AUTOMATICAMENTE]
```

### Limitações Atuais
- Features limitadas (apenas GDP, renda, ano, quantidade de países)
- Dados agregados por delegação (perda de granularidade por esporte)
- Ausência de features históricos (ex: performance passada)
- Modelo de regressão linear (não captura dinâmicas de contagem/zero-inflação)

### Próximos Passos Recomendados

1. **Enriquecimento de Features:**
   - Adicionar nº de atletas, investimento esportivo, população, PIB per capita
   - Criar features de performance histórica (medalhas nos últimos ciclos olímpicos)
   - Features de região geográfica

2. **Granularidade por Esporte:**
   - Prever medalhas por delegação × esporte
   - Agregar depois para capturar heterogeneidade

3. **Modelos de Contagem:**
   - Poisson / Negative Binomial Regression
   - Zero-Inflated Models (para lidar com zeros excessivos)

4. **Validação Temporal Robusta:**
   - Rolling time-splits (validação em múltiplas janelas)
   - Expanding window validation

5. **Otimização Avançada:**
   - Busca Bayesiana (Optuna) em vez de RandomizedSearchCV
   - Stacking / Ensembles de múltiplos modelos

6. **Incerteza e Intervalos de Confiança:**
   - Quantile Regression
   - Conformal Prediction
   - Modelos Bayesianos

7. **Explainability:**
   - SHAP (SHapley Additive exPlanations)
   - Partial Dependence Plots (PDPs)
   - Análise de erros por país/região

---

**Autor:** Thiago Luiz Metne  
**Email:** thiagol.metne@gmail.com